# 第 8 章 · 记忆（ADK：用工具改 state）

`session.state` 是这次会话的临时数据。**改它有三条路**，这是第一条：**在工具里改**。

为什么推荐用工具？因为它把"改状态"这件事**收在一个可测试的函数里**，
而不是散落在 instruction 的自然语言里（`LlmAgent` 自己根本写不了 state）。

kernel 选 `Python (agentic-adk)`。

In [1]:
# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）

# 通用脚手架都在 common 里：run_once / run_session / run_async / 重试 / 静音
from common import (adk_model, quiet_adk_sync_warnings, retry_call,
                    run_once, run_session, run_async, require_adk)
quiet_adk_sync_warnings()

require_adk()      # 【适配】内核选错时给出可操作提示（内部就是 import google.adk）

from google.adk.agents import LlmAgent
from google.adk.tools import ToolContext


def log_user_login(tool_context: ToolContext) -> dict:
    """Updates the session state upon a user login event.

    Args:
        tool_context: Automatically provided by ADK, gives access to session state.

    Returns:
        A dictionary confirming the action was successful.
    """
    # ★ 本章新增：tool_context.state 是**可写**的 —— 工具是改 state 的正规入口
    tool_context.state["last_login_user"] = "alice"
    tool_context.state["login_count"] = tool_context.state.get("login_count", 0) + 1
    return {"status": "ok", "user": "alice"}


login_agent = LlmAgent(
    name="LoginAgent",
    model=adk_model(),                      # 【适配】书里没给 model
    instruction=
        "用户说要登录时，调用 log_user_login 工具，然后把返回的 status 复述一遍，并返回最后登录账号。",
    tools=[log_user_login],
)

finals, state = run_once(login_agent, "我要登录")
print("agent 答复:", finals[-1][1] if finals else "(无)")
print("\nsession.state =", {k: v for k, v in state.items()})

agent 答复: 状态：ok
最后登录账号：alice

session.state = {'last_login_user': 'alice', 'login_count': 1}
